In [1]:
import pandas as pd
import numpy as np
import ast
import re
from collections import Counter
import json

import preprocessing as pp
import candidates as ca
import refinements as rf
import subgroup_selection as ss
import search as bs
import search_storing as store_bs
import dfd as dfd
import fpgrowth as fp

In [2]:
unclean_data = pd.read_csv('Data/algebrakit_firstbatch_2.csv')
df = pp.clean_data_algebrakit(unfilterd_data=unclean_data, filter_type="remove_interaction")
df['representative_index'] = range(len(df))
print(f"we have {len(df)} records of {len(df['sessionID'].unique())} sessions with {df['finished'].value_counts()}")

we have 6020 records of 1622 sessions with finished
False    3958
True     2062
Name: count, dtype: int64


In [3]:
ids = ['exerciseID', 'sessionID', 'questionID', 'interactionID', 'representative_index']

df_test = df[ids + ['skills']]

df_test.head()

,exerciseID,sessionID,questionID,interactionID,representative_index,skills
0,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,Q1,Ex8wE,0,[FractionArithmeticMultiplicationProper]
1,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,Q2,gBslt,1,[FractionArithmeticMultiplicationProper]
2,9f4a675e-c2b3-4e11-91cc-4445951bab60,03bba4c5-afea-42c0-87bb-f99e5f2d2bfb,Q1,Ex8wE,2,[FractionArithmeticMultiplicationProper]
3,9f4a675e-c2b3-4e11-91cc-4445951bab60,040b59bb-b5da-4ef0-ba5b-3e23f163c7de,Q1,Ex8wE,3,[FractionArithmeticMultiplicationProper]
4,9f4a675e-c2b3-4e11-91cc-4445951bab60,040b59bb-b5da-4ef0-ba5b-3e23f163c7de,Q2,gBslt,4,[FractionArithmeticMultiplicationProper]


In [4]:
import random
idx = random.sample(range(3000), 354)

FIs = fp.fpgrowth(subgroup=df_test.iloc[idx], attribute_of_interest='skills', closed=True)
                
i = 0
for item, info in FIs.items():
    if info['frequency'] > 20:
        i += 1
        print(f"({i})")
        print(f"Collection: {list(item)}")
        print(f"Frequency: {info['frequency']}")
        # print(f"Indices: {info['indices']}")
        print(f"= = = = = = = = = = = = = = =")

(1)
Collection: ['FractionArithmeticSimilarImproperToMixed', 'FractionArithmeticSimplifyProper']
Frequency: 48
= = = = = = = = = = = = = = =
(2)
Collection: ['EquationsBalanceMethodDivideCoefficient', 'FractionArithmeticSimplifyProper', 'EquationsBalanceMethodArrangeTerms', 'EquationsOperationsMultiply', 'EquationsOperationsAdd']
Frequency: 28
= = = = = = = = = = = = = = =
(3)
Collection: ['EquationsOperationsAdd', 'EquationsBalanceMethodArrangeTerms']
Frequency: 94
= = = = = = = = = = = = = = =
(4)
Collection: ['EquationsOperationsMultiply', 'EquationsBalanceMethodArrangeTerms', 'EquationsBalanceMethodDivideCoefficient', 'EquationsOperationsAdd']
Frequency: 87
= = = = = = = = = = = = = = =
(5)
Collection: ['AlgebraAddingLikeTerms', 'EquationsBalanceMethodDivideCoefficient', 'EquationsBalanceMethodArrangeTerms', 'EquationsOperationsMultiply', 'EquationsOperationsAdd']
Frequency: 50
= = = = = = = = = = = = = = =
(6)
Collection: ['EquationsOperationsMultiply', 'AlgebraAddingLikeTerms']
F

In [5]:
import copy
FIs_test = fp.fpgrowth(subgroup=df_test.iloc[idx], attribute_of_interest='skills', closed=True)

sg_idx = df_test.iloc[idx]['representative_index']
FI_pruned = fp.prune_frequent_itemsets(FI=FIs_test, sg_indices=sg_idx)
                
i = 0
for item, info in FI_pruned.items():
    if info['frequency'] > 20:
        i += 1
        print(f"({i})")
        print(f"Collection: {list(item)}")
        print(f"Frequency: {info['frequency']}")
        # print(f"Indices: {info['indices']}")
        print(f"= = = = = = = = = = = = = = =")

(1)
Collection: ['FractionArithmeticSimilarImproperToMixed', 'FractionArithmeticSimplifyProper']
Frequency: 48
= = = = = = = = = = = = = = =
(2)
Collection: ['EquationsBalanceMethodDivideCoefficient', 'FractionArithmeticSimplifyProper', 'EquationsBalanceMethodArrangeTerms', 'EquationsOperationsMultiply', 'EquationsOperationsAdd']
Frequency: 28
= = = = = = = = = = = = = = =
(3)
Collection: ['EquationsBalanceMethodArrangeTerms', 'EquationsOperationsAdd']
Frequency: 94
= = = = = = = = = = = = = = =
(4)
Collection: ['EquationsOperationsMultiply', 'EquationsOperationsAdd', 'EquationsBalanceMethodDivideCoefficient', 'EquationsBalanceMethodArrangeTerms']
Frequency: 87
= = = = = = = = = = = = = = =
(5)
Collection: ['AlgebraAddingLikeTerms', 'EquationsBalanceMethodDivideCoefficient', 'EquationsBalanceMethodArrangeTerms', 'EquationsOperationsMultiply', 'EquationsOperationsAdd']
Frequency: 50
= = = = = = = = = = = = = = =
(6)
Collection: ['EquationsOperationsMultiply', 'AlgebraAddingLikeTerms']
F